In [ ]:
import dotenv

dotenv.load_dotenv()
import random

import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from IPython.display import display

import newaresql

You must either set the enviroment variables
- `NEWARE_BTS_HOST`
- `NEWARE_BTS_PORT`
- `NEWARE_BTS_USERNAME`
- `NEWARE_BTS_PASSWORD`
- `NEWARE_BTS_DATABASE`
- 
globally or in the env file, modify the credentials dictionary

In [ ]:
credentials = dict(
    host=None,
    port=None,
    username=None,
    password=None,
    database=None,
)

In [ ]:
with newaresql.connect(**credentials) as conn:
    tests = conn.get_tests()
    print(f"Found {len(tests)} tests in the database")
    display(tests.head())


In [ ]:
with newaresql.connect(**credentials) as conn:
    test = random.choice(tests.to_dicts())
    data = conn.get_data(test, conditions={"cycle": (None, 3)})

In [ ]:
fig = sp.make_subplots(rows=4, cols=1)
merged = data.data
for n, col in enumerate(
    ["Current / A", "Voltage / V", "Power / W", "Temperature / degC"], start=1
):
    fig.add_trace(
        go.Scatter(
            x=merged.select("Unix Time / s").to_series(),
            y=merged.select((col,)).to_series(),
            name=col,
        ),
        row=n,
        col=1,
    )
    fig.update_xaxes(title_text="Unix Time / s", row=n, col=1)
    fig.update_yaxes(title_text=col, row=n, col=1)
fig.update_layout(width=1000, height=1000)
fig.show()
